# Load PyTorch Chunk Files (`.pt`)

This notebook demonstrates how to load and inspect mixed human/AI datasets saved by `main.py`.

Each `.pt` file contains:
- `texts` — full mixed text
- `labels` — word-level labels (`0` = human, `1` = AI)
- `models` — OpenRouter model used per sample
- `indices` — original MiniPile index

In [11]:
from pathlib import Path

from load_pt import (
    MixedTextChunkDataset,
    chunks_in_range,
    collate_mixed_batch,
    list_chunks,
    load_chunk,
    load_split,
    make_dataloader,
    summarize_chunk,
)

OUTPUT_DIR = Path("output")
SPLIT = "train"

## 1. List available chunk files

In [12]:
chunks = list_chunks(OUTPUT_DIR, SPLIT)
print(f"Found {len(chunks)} chunk(s) for split '{SPLIT}':")
for path in chunks:
    print(f"  {path}")

Found 1 chunk(s) for split 'train':
  output/train/train_0_999.pt


## 2. Compare with original text

This cell loads one mixed sample, fetches the original MiniPile text by index,
and compares:
- original text
- recovered human part (labels == 0)
- recovered AI part (labels == 1)

In [ ]:
from pathlib import Path

import pyarrow.parquet as pq

PT_PATH = chunks[0] if chunks else OUTPUT_DIR / SPLIT / "train_0_999.pt"
DATA_DIR = Path("dataset/minipile/data")

info = summarize_chunk(PT_PATH)
print(f"Path:       {info['path']}")
print(f"Samples:    {info['samples']}")
print(f"Indices:    {info['index_range'][0]}–{info['index_range'][1]}")
print(f"Model set:  {', '.join(info['models'])}")

def get_original_text(data_dir: Path, split: str, index: int) -> str:
    files = sorted(data_dir.glob(f"{split}-*.parquet"))
    seen = 0
    for file_path in files:
        table = pq.read_table(file_path, columns=["text"])
        n = table.num_rows
        if seen + n > index:
            return table.column("text")[index - seen].as_py()
        seen += n
    raise IndexError(f"Index {index} out of range for split '{split}'")

Path:       output/train/train_0_999.pt
Samples:    1000
Indices:    0–999
Model set:  google/gemma-4-26b-a4b-it:free, liquid/lfm-2.5-1.2b-instruct:free, minimax/minimax-m2.5:free, nvidia/nemotron-3-super-120b-a12b:free, nvidia/nemotron-nano-12b-v2-vl:free, openai/gpt-oss-120b:free, openai/gpt-oss-20b:free, openrouter/free, openrouter/owl-alpha


## 3. Load one chunk directly

In [28]:
data = load_chunk(PT_PATH)

print("Keys:", list(data.keys()))
print("Sample count:", len(data["texts"]))

Keys: ['texts', 'labels', 'models', 'indices']
Sample count: 1000


In [35]:

from random import randint


TARGET_IN_CHUNK = randint(0, len(data["texts"]) - 1)  # change this to compare a different sample inside the chunk
item = MixedTextChunkDataset(PT_PATH)[TARGET_IN_CHUNK]
print(item)
sample_index = item["index"]
original_text = get_original_text(DATA_DIR, SPLIT, sample_index)
words = item["text"].split()
labels = item["labels"].tolist()
human_length = sum(1 for l in labels if l == 0)
human_part = " ".join(words[:human_length])
ai_part = " ".join(words[human_length:])

print(f"\nSample index: {sample_index}")
print(f"Model used:    {item['model']}")
print("\n--- Original text[:human_length]")
print(" ".join(original_text.split()[:human_length]))
print("\n--- Original text[human_length:]")
print(" ".join(original_text.split()[human_length:]))
print("\n--- Human part recovered from labels")
print(human_part)
print("\n--- AI continuation")
print(ai_part)
print("\n--- labels")
print(labels)

{'text': 'In paging systems, subscribers receive page messages, or pages, that have been sent by users of the paging system. Generally, pages only go to the subscribers that they are intended for, and likewise, subscribers only get the pages that are intended for them. A unique use of paging systems arises when third party subscribers, or message monitoring users, wish to obtain pages that receiving such pages can violate privacy, which is why most modern systems bind each message to an encrypted header only accessible to the intended recipient. Even though the infrastructure technically supports wide broadcast, regulatory safeguards limit exposure to authorized devices.', 'labels': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

## 4. Use `MixedTextChunkDataset`

In [10]:
ds = MixedTextChunkDataset(PT_PATH)
sample = ds[0]

print(f"Dataset size: {len(ds)}")
print(f"Index:  {sample['index']}")
print(f"Model:  {sample['model']}")
print(f"Text:   {sample['text']}")
print(f"Labels: {sample['labels'].tolist()}")

Dataset size: 1000
Index:  0
Model:  openrouter/owl-alpha
Text:   HTC's Vive Pro headset is available to pre-order for $799 We've seen plenty of Beats-focused KIRFs in our time, some better than others. Few, however, play quite so directly on the name as OrigAudio's Beets. For $25, adopters get a set of headphones that bear little direct resemblance to Dr. Dre's audio gear of choice, but are no doubt bound to impress friends -- at least, up until they see a root vegetable logo instead of a lower-case B. Thankfully, there's more to it than just amusing and confusing peers. Every pair delivers surprisingly rich audio quality that belies its playful appearance. The sound profile leans slightly warm, with crisp highs and punchy bass response. Battery life stretches comfortably through several hours of regular listening. These quirky headphones prove style and substance can coexist beautifully together in one package.
Labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

## 5. Load a split with index range filter

Load only samples in `[from_idx, to_idx)`.

In [6]:
FROM_IDX = 0
TO_IDX = 100

subset = load_split(OUTPUT_DIR, SPLIT, from_idx=FROM_IDX, to_idx=TO_IDX)
print(f"Subset size: {len(subset)}")
print(f"First index: {subset[0]['index']}")
print(f"Last index:  {subset[-1]['index']}")

Subset size: 100
First index: 0
Last index:  99


## 6. DataLoader with padded labels

Labels are padded with `-1` for batching (variable-length sequences).

In [7]:
loader = make_dataloader(
    OUTPUT_DIR,
    SPLIT,
    batch_size=4,
    shuffle=False,
    from_idx=FROM_IDX,
    to_idx=TO_IDX,
)

batch = next(iter(loader))
print("Batch keys:", batch.keys())
print("Texts:", len(batch["text"]))
print("Labels shape:", batch["labels"].shape)
print("Indices:", batch["index"].tolist())
print("Models:", batch["model"])

Batch keys: dict_keys(['text', 'labels', 'model', 'index'])
Texts: 4
Labels shape: torch.Size([4, 144])
Indices: [0, 1, 2, 3]
Models: ['openrouter/owl-alpha', 'openrouter/owl-alpha', 'openrouter/owl-alpha', 'openai/gpt-oss-20b:free']


## 7. Inspect human vs AI word labels

Visualize which words are human (`0`) vs AI (`1`).

In [8]:
def show_labeled_words(text: str, labels):
    words = text.split()
    labeled = [(word, int(label)) for word, label in zip(words, labels.tolist())]
    human = [w for w, l in labeled if l == 0]
    ai = [w for w, l in labeled if l == 1]
    print(f"Human words ({len(human)}): {' '.join(human[:20])}{'...' if len(human) > 20 else ''}")
    print(f"AI words    ({len(ai)}): {' '.join(ai[:20])}{'...' if len(ai) > 20 else ''}")


item = subset[0]
show_labeled_words(item["text"], item["labels"])

Human words (91): HTC's Vive Pro headset is available to pre-order for $799 We've seen plenty of Beats-focused KIRFs in our time, some...
AI words    (48): pair delivers surprisingly rich audio quality that belies its playful appearance. The sound profile leans slightly warm, with crisp highs...
